In [30]:
import pandas as pd
import numpy as np
import json
from datetime import timedelta

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

import warnings
warnings.filterwarnings("ignore")

In [31]:
inventory_df = pd.read_csv("inventory_transactions.csv")
production_df = pd.read_csv("production_orders.csv")
supplier_df = pd.read_csv("supplier_master.csv")
material_df = pd.read_csv("material_master.csv")
working_capital_df = pd.read_csv("working_capital_log.csv")
seasonal_df = pd.read_csv("seasonal_index.csv")

In [32]:
inventory_df.columns = inventory_df.columns.str.strip()
production_df.columns = production_df.columns.str.strip()
supplier_df.columns = supplier_df.columns.str.strip()
material_df.columns = material_df.columns.str.strip()
working_capital_df.columns = working_capital_df.columns.str.strip()
seasonal_df.columns = seasonal_df.columns.str.strip()

In [33]:
print(material_df.columns.tolist())

['material_id', 'name', 'category', 'unit', 'reorder_point_current', 'current_stock', 'warehouse_location', 'substitute_material_ids']


In [34]:
print(inventory_df.head())
print(production_df.head())
print(material_df.head())

         date material_id transaction_type  quantity   unit supplier_id  \
0  2022-01-01         M08            issue     63.86     kg         NaN   
1  2022-01-01         M01            issue     21.99  rolls         NaN   
2  2022-01-01         M02           return     54.35  rolls         NaN   
3  2022-01-01         M09            issue     76.63     kg         NaN   
4  2022-01-01         M13           return     50.03  rolls         NaN   

   unit_price po_number  
0     7382.83       NaN  
1    13183.04       NaN  
2    30618.76       NaN  
3    28773.91       NaN  
4    31516.23       NaN  
       order_id           client_id    product_type box_size  quantity  \
0  PO-2022-0000     Godrej Consumer       small_std    small      9281   
1  PO-2022-0001  Hindustan Unilever      medium_std   medium     54211   
2  PO-2022-0002        Nestle India       small_std    small      9621   
3  PO-2022-0003           Britannia      medium_std   medium     49895   
4  PO-2022-0004        

In [35]:
inventory_df['date'] = pd.to_datetime(inventory_df['date'])
production_df['delivery_date'] = pd.to_datetime(production_df['delivery_date'])

In [36]:
# Example conversion map
conversion_map = {
    'kg': 1,
    'roll': 50,
    'rolls': 50,
    'piece': 0.5,
    'pieces': 0.5
}

inventory_df['normalized_quantity'] = inventory_df.apply(
    lambda x: x['quantity'] * conversion_map.get(str(x['unit']).lower(), 1),
    axis=1
)

In [37]:
production_df['material_bom'] = production_df['material_bom'].apply(json.loads)

In [38]:
material_demand = []

for _, row in production_df.iterrows():

    quantity = row['quantity']
    delivery_date = row['delivery_date']

    bom = row['material_bom']

    for material_id, qty_per_unit in bom.items():

        total_material_needed = quantity * qty_per_unit

        material_demand.append({
            'delivery_date': delivery_date,
            'material_id': material_id,
            'required_qty': total_material_needed
        })

demand_df = pd.DataFrame(material_demand)

In [39]:
weekly_demand = demand_df.groupby(
    ['delivery_date', 'material_id']
)['required_qty'].sum().reset_index()

weekly_demand.head()

,delivery_date,material_id,required_qty
0,2022-01-18,M04,32668.900
1,2022-01-18,M05,22571.240
2,2022-01-18,M07,1306.756
3,2022-01-18,M09,475.184
4,2022-01-18,M10,475.184


In [40]:
weekly_demand['month'] = weekly_demand['delivery_date'].dt.month

weekly_demand = weekly_demand.merge(
    seasonal_df,
    on='month',
    how='left'
)

weekly_demand['seasonal_demand'] = (
    weekly_demand['required_qty'] *
    weekly_demand['fmcg_demand_multiplier']
)

In [41]:
weekly_demand['day'] = weekly_demand['delivery_date'].dt.day
weekly_demand['week'] = weekly_demand['delivery_date'].dt.isocalendar().week
weekly_demand['year'] = weekly_demand['delivery_date'].dt.year

In [42]:
weekly_demand['material_code'] = (
    weekly_demand['material_id']
    .astype('category')
    .cat.codes
)

In [43]:
features = [
    'month',
    'day',
    'week',
    'year',
    'material_code',
    'fmcg_demand_multiplier'
]

X = weekly_demand[features]
y = weekly_demand['seasonal_demand']

In [44]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [45]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [46]:
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)

print("MAE:", mae)

MAE: 4917.75359689057


In [47]:
future_predictions = weekly_demand.copy()

future_predictions['predicted_demand'] = model.predict(
    future_predictions[features]
)

future_predictions.head()

,delivery_date,material_id,required_qty,month,month_name,fmcg_demand_multiplier,notes,seasonal_demand,day,week,year,material_code,predicted_demand
0,2022-01-18,M04,32668.900,1,Jan,0.78,Post-festival lull,25481.74200,18,3,2022,2,21842.347027
1,2022-01-18,M05,22571.240,1,Jan,0.78,Post-festival lull,17605.56720,18,3,2022,3,18453.739424
2,2022-01-18,M07,1306.756,1,Jan,0.78,Post-festival lull,1019.26968,18,3,2022,5,979.865268
3,2022-01-18,M09,475.184,1,Jan,0.78,Post-festival lull,370.64352,18,3,2022,7,346.159022
4,2022-01-18,M10,475.184,1,Jan,0.78,Post-festival lull,370.64352,18,3,2022,8,333.514025


In [48]:
current_inventory = inventory_df.groupby(
    'material_id'
)['normalized_quantity'].sum().reset_index()

current_inventory.columns = [
    'material_id',
    'available_stock'
]

current_inventory.head()

,material_id,available_stock
0,M01,5241873.50
1,M02,4996062.44
2,M03,5493694.37
3,M04,4747616.83
4,M05,5089071.75


In [49]:
inventory_status = material_df.merge(
    current_inventory,
    on='material_id',
    how='left'
)

inventory_status['available_stock'] = (
    inventory_status['available_stock']
    .fillna(0)
)

In [50]:
forecast_summary = future_predictions.groupby(
    'material_id'
)['predicted_demand'].sum().reset_index()

procurement_df = inventory_status.merge(
    forecast_summary,
    on='material_id',
    how='left'
)

procurement_df['predicted_demand'] = (
    procurement_df['predicted_demand']
    .fillna(0)
)

procurement_df['recommended_order_qty'] = (
    procurement_df['predicted_demand']
    - procurement_df['available_stock']
)

procurement_df['recommended_order_qty'] = (
    procurement_df['recommended_order_qty']
    .apply(lambda x: max(0, x))
)

In [51]:
procurement_df = procurement_df.merge(
    supplier_df[['supplier_id', 'moq']],
    how='left',
    left_index=True,
    right_index=True
)

procurement_df['final_order_qty'] = procurement_df.apply(
    lambda x:
    np.ceil(x['recommended_order_qty'] / x['moq']) * x['moq']
    if pd.notnull(x['moq']) and x['moq'] > 0
    else x['recommended_order_qty'],
    axis=1
)

In [52]:
latest_prices = inventory_df.groupby(
    'material_id'
)['unit_price'].mean().reset_index()

procurement_df = procurement_df.merge(
    latest_prices,
    on='material_id',
    how='left'
)

procurement_df['estimated_cost'] = (
    procurement_df['final_order_qty'] *
    procurement_df['unit_price']
)

In [53]:
latest_capital = working_capital_df.iloc[-1]

available_credit = latest_capital['available_credit_inr']

procurement_df = procurement_df.sort_values(
    by='estimated_cost'
)

selected_orders = []
running_total = 0

for _, row in procurement_df.iterrows():

    cost = row['estimated_cost']

    if running_total + cost <= available_credit:

        selected_orders.append(row)

        running_total += cost

final_orders_df = pd.DataFrame(selected_orders)

In [58]:
procurement_df['days_of_stock'] = np.where(
    procurement_df['predicted_demand'] > 0,
    procurement_df['available_stock'] /
    (procurement_df['predicted_demand'] / 28),
    np.inf
)
stockout_alerts = procurement_df[
    procurement_df['days_of_stock'] < 3
]


stockout_alerts[
    ['material_id', 'days_of_stock']
]

,material_id,days_of_stock


In [59]:
print(procurement_df[
    ['material_id',
     'available_stock',
     'predicted_demand',
     'days_of_stock']
].head(20))

   material_id  available_stock  predicted_demand  days_of_stock
2          M03       5493694.37      0.000000e+00            inf
11         M12        108685.59      4.896798e+04      62.146666
12         M13       4945592.12      2.675660e+06      51.754176
13         M14       4799205.03      2.698140e+06      49.803838
10         M11        117819.73      1.569135e+05      21.024019
9          M10        108438.38      1.868178e+05      16.252598
8          M09        105604.17      1.853970e+05      15.949106
6          M07        113703.47      2.542883e+05      12.520028
5          M06        111865.25      6.689558e+05       4.682263
1          M02       4996062.44      5.564290e+06      25.140627
7          M08        105694.70      8.361507e+05       3.539376
3          M04       4747616.83      6.117837e+06      21.728804
0          M01       5241873.50      1.083496e+07      13.546199
4          M05       5089071.75      1.545705e+07       9.218704


In [56]:
critical_materials = stockout_alerts.merge(
    material_df[
        ['material_id', 'substitute_material_ids']
    ],
    on='material_id',
    how='left'
)

critical_materials[
    ['material_id', 'substitute_material_ids']
]

KeyError: "['substitute_material_ids'] not in index"

In [ ]:
report_columns = [
    'material_id',
    'name',
    'available_stock',
    'predicted_demand',
    'final_order_qty',
    'estimated_cost'
]

weekly_report = procurement_df[report_columns]

weekly_report.to_excel(
    "Weekly_Purchase_Recommendation.xlsx",
    index=False
)

print("Report Generated Successfully")

In [ ]:
import joblib

joblib.dump(model, "inventory_forecasting_model.pkl")

print("Model Saved")